# Data Reading from Bronze Layer

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/03.Silver_Helper


In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
sliver_table = f"{catalog_name}.{silver_schema}.circuits"

In [0]:
sliver_table

In [0]:
from pyspark.sql.functions import*

In [0]:
circuits_df= spark.read.table(bronze_table).filter(col('batch_id') == v_batch_id)



# Data Cleaning

In [0]:
circuits_selected_df=circuits_df.select(
                                        col("circuitId"),
                                        col("circuitName"),
                                        col("lat"),
                                        col("long"),
                                        col("locality"),
                                        col("country"),
                                        col("ingestion_timestamp"),
                                        col("source_files"),
                                        col("batch_id")
                                    )

In [0]:
#circuits_renamed_df=(circuits_selected_df.withColumnRenamed("circuitId","circuit_id") 
                  # .withColumnRenamed("circuitName","circuit_name") 
                   #.withColumnRenamed("lat","latitude") 
                   #.withColumnRenamed("long","longitude") 
                   #.withColumnRenamed("locality","city") 
                   #)
#circuits_renamed_df.show(5)

In [0]:
circuits_renamed_df = circuits_selected_df.withColumnsRenamed(
   {
        "circuitId": "circuit_id",
       "circuitName": "circuit_name",
       "lat": "latitude",
       "long": "longitude",
       "locality": "city"
   }
)

In [0]:
circuits_renamed_df.show(5)

# Data Validation

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    col("circuit_id").isNotNull()
)

# Removing duplicates Circuit ID

In [0]:
circuits_distincts_df = circuits_valid_df.dropDuplicates(["circuit_id"])

# Making Circuit Name and City in INITCAP

In [0]:
circuits_final_df = (
                    circuits_distincts_df
                     .withColumn("circuit_name", initcap(col("circuit_name")))
                     .withColumn("city", initcap(col("city")))
                    )

# Write Data as Delta Table in Silver Schema

Databricks Delta tables mein older Bronze data se Silver table ko overwrite hone se bachane ke liye MERGE INTO + timestamp/version/batch_id check use kar sakte hain.
Example: maan lo Bronze table mein customer data aa raha hai, aur Silver mein latest cleaned data maintain karna hai.
- Bronze mein customer ka purana record dobara aa sakta hai.
- Silver mein us customer ka newer record already hai.
- Humein purane record se Silver ko update nahi karna chahiye.
- Kyoki silver me already new data hai usay purane overwrite krke datamiss krdegey

In [0]:
# if not spark.catalog.tableExists(sliver_table):
#   circuits_final_df.write.mode("overwrite").format("delta").saveAsTable(sliver_table)

# else:
#     delta_table = DeltaTable.forName(spark, sliver_table)  
    

#     (
#         delta_table.alias('t')
#         .merge(circuits_final_df.alias('s'), "t.circuit_id=s.circuit_id")
#         .whenMatchedUpdate(
#             condition="s.batch_id>=t.batch_id",
#             set={
#                 "t.circuit_name":"s.circuit_name",
#                 "t.latitude":"s.latitude",
#                 "t.longitude":"s.longitude",
#                 "t.city":"s.city",
#                 "t.ingestion_timestamp":"s.ingestion_timestamp",
#                 "t.source_files":"s.source_files",
#                 "t.batch_id":"s.batch_id",
#                 "t.updated_timestamp":"s.updated_timestamp"
#             }

#         )
#         .whenNotMatchedInsertAll()
#         .execute()
#     )

In [0]:
write_to_silver(input_df=circuits_final_df,                                         target_table=sliver_table,
                merge_condn="t.circuit_id=s.circuit_id",
                columns_to_update=["circuit_name",
                                    "latitude",
                                    "longitude",
                                    "city",
                                    "country",
                                    "ingestion_timestamp",
                                    "source_files",
                                    "batch_id"
                                ]
                )

In [0]:
display(spark.table(sliver_table))